# 06 · Chunked Prefill 与尾延迟

前两章解决**吞吐**问题。这一章解决**延迟**，而且是延迟里最要命的那个指标：**尾延迟（p99）**。

场景很常见：服务正在稳定处理一批 decode 请求，突然来了一条 4K token 的长 prompt。会发生什么？

In [ ]:
# ===== 引导单元：环境检查 + 测量工具 + MiniGPT（每章自带，直接运行）=====
# 说明：本单元在每个 notebook 里都有一份完整副本，目的是让任何一个 notebook
#       都能在 Colab 里零配置独立运行。想改模型结构，请改 tools/build_notebooks.py
#       里的 SETUP_CODE，然后重跑编译脚本。
import math
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# MiniGPT 只有 2700 万参数，用 float16 跑在 GPU 上；CPU 上 float16 很慢，用 float32
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32


def sync():
    """GPU 是异步执行的，计时前必须同步，否则测到的是下发时间不是执行时间。"""
    if DEVICE == "cuda":
        torch.cuda.synchronize()


def bench(fn, warmup=3, iters=10):
    """返回单次调用的平均耗时（毫秒）。warmup 用来排除首次 kernel 编译等开销。"""
    for _ in range(warmup):
        fn()
    sync()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    sync()
    return (time.perf_counter() - t0) / iters * 1000.0


def peak_mem_mb():
    """当前 CUDA 峰值显存占用（MB）。"""
    if DEVICE != "cuda":
        return 0.0
    return torch.cuda.max_memory_allocated() / 1024 ** 2


def reset_peak():
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()


class Config:
    def __init__(self, vocab_size=50257, block_size=1024, n_layer=4, n_head=6, n_embd=384):
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_embd = n_embd
        self.head_dim = n_embd // n_head


class CausalSelfAttention(nn.Module):
    """因果自注意力，支持 KV cache。

    past_kv 传入历史的 (k, v)，本步只为新 token 计算 Q/K/V，然后拼在历史后面。
    返回 (输出, 更新后的 (k, v))，其中 k/v 的 shape 是 (B, n_head, 总长度, head_dim)。
    """

    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.head_dim = cfg.head_dim
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)

    def forward(self, x, past_kv=None, attn_mask=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)

        S = k.size(2)  # 总长度 = 历史 + 本步新增
        if attn_mask is None:
            # 默认因果掩码：本步第 i 个 query 的绝对位置是 S-T+i，只能看见 <= 它的 key
            mask = torch.ones(T, S, device=x.device).tril(diagonal=S - T).bool()
        else:
            # 外部传入的掩码，用于一个 batch 里混合不同进度的序列（第 04、06 章）
            mask = attn_mask
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y), (k, v)


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc = nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=False)

    def forward(self, x):
        return self.proj(F.gelu(self.fc(x)))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = MLP(cfg)

    def forward(self, x, past_kv=None, attn_mask=None):
        h, present = self.attn(self.ln_1(x), past_kv, attn_mask)
        x = x + h
        x = x + self.mlp(self.ln_2(x))
        return x, present


class MiniGPT(nn.Module):
    """极简 GPT，结构与 Llama 同源：pre-norm + 因果注意力 + 4 倍扩张 MLP + 权重共享。

    与 Llama 的两处差异：
      - 用可学习位置编码代替 RoPE（简化实现，不影响调度实验的结论）
      - 没有 GQA（本仓库是 MHA，第 03 章会手工比较两者的 KV cache 大小）
    """

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.wpe = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight  # 权重共享，省一份 embedding 参数

        def init(m):
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)

        self.apply(init)

    def forward(self, idx, past_kvs=None, pos_offset=0, attn_mask=None):
        """idx: (B, T) 的 token id。

        past_kvs: 长度等于层数的列表，每项是 (k, v)；None 表示从零开始（prefill）。
        pos_offset: 本次输入的第一个 token 的绝对位置。传 int 表示整个 batch 用同一个
                    偏移；传 shape (B,) 的张量表示每条序列各用各的偏移——当 batch 里
                    混合了不同进度的请求时必须这样传。
        attn_mask: 可选的自定义注意力掩码，用于屏蔽填充位。
        """
        B, T = idx.shape
        if torch.is_tensor(pos_offset):
            pos = pos_offset.view(B, 1) + torch.arange(T, device=idx.device)[None, :]
        else:
            pos = torch.arange(pos_offset, pos_offset + T, device=idx.device)[None, :].expand(B, T)
        x = self.wte(idx) + self.wpe(pos)

        presents = []
        for i, blk in enumerate(self.blocks):
            past = None if past_kvs is None else past_kvs[i]
            x, present = blk(x, past, attn_mask)
            presents.append(present)
        return self.lm_head(self.ln_f(x)), presents

    @property
    def n_params(self):
        return sum(p.numel() for p in self.parameters())


def build_model(seed=0, device=DEVICE, dtype=DTYPE, **kw):
    torch.manual_seed(seed)
    cfg = Config(**kw)
    model = MiniGPT(cfg).to(device=device, dtype=dtype)
    return model.eval()


@torch.no_grad()
def generate_naive(model, idx, max_new_tokens):
    """不用 KV cache：每一步都把完整序列重新算一遍（O(n^2) 重算）。"""
    for _ in range(max_new_tokens):
        logits, _ = model(idx[:, -model.cfg.block_size:])
        idx = torch.cat([idx, logits[:, -1].argmax(-1, keepdim=True)], dim=1)
    return idx


@torch.no_grad()
def generate_cached(model, idx, max_new_tokens):
    """用 KV cache：prompt 只 prefill 一次，之后每步只喂 1 个 token。"""
    logits, past = model(idx)
    nxt = logits[:, -1].argmax(-1, keepdim=True)
    out = [nxt]
    pos = idx.size(1)
    for _ in range(max_new_tokens - 1):
        logits, past = model(nxt, past_kvs=past, pos_offset=pos)
        pos += 1
        nxt = logits[:, -1].argmax(-1, keepdim=True)
        out.append(nxt)
    return torch.cat([idx] + out, dim=1)


def kv_bytes(n_layer, n_kv_head, head_dim, seq_len, batch=1, dtype_bytes=2):
    """KV cache 字节数。注意是 2（K 和 V 各一份）。"""
    return 2 * n_layer * n_kv_head * head_dim * seq_len * batch * dtype_bytes


print(f"引导单元加载完成 | device={DEVICE} dtype={DTYPE} torch={torch.__version__}")
# ===== 引导单元结束 =====

## 一、先看问题的严重程度

长 prefill 是一次无法中断的大矩阵乘法。在它执行期间 GPU 被它独占，其他请求只能等着。这就是**队头阻塞（head-of-line blocking）**。

In [ ]:
model = build_model(block_size=4096)

# 一个 decode step（batch=8，上下文 128）
dec_ids = torch.randint(0, model.cfg.vocab_size, (8, 128), device=DEVICE)
_, dec_past = model(dec_ids)
dec_next = torch.randint(0, model.cfg.vocab_size, (8, 1), device=DEVICE)
dec_ms = bench(lambda: model(dec_next, past_kvs=dec_past, pos_offset=128), warmup=3, iters=10)

print(f"单个 decode step（batch=8）: {dec_ms:.1f} ms\n")
for L in [256, 1024, 2048]:
    long_ids = torch.randint(0, model.cfg.vocab_size, (1, L), device=DEVICE)
    ms = bench(lambda: model(long_ids), warmup=2, iters=5)
    print(f"prefill {L:>5} token : {ms:>8.1f} ms   （是单个 decode step 的 {ms / dec_ms:>5.1f} 倍）")

print()
print("一条 2048 token 的 prefill，会让其他正在解码的请求多等几十到上百毫秒。")
print("用户视角就是'打字打到一半卡住了'。")

## 二、Chunked Prefill 的做法

思路：**把长 prefill 切成若干小块，每轮迭代只算一块，和该轮的 decode 请求拼在一起执行。**

关键认知：**这不会让总计算量减少。** 它只是把一次很长的阻塞，摊成多次很短的阻塞。

实现上，借助 `past_kvs` 和 `pos_offset` 就能连续分块：

In [ ]:
@torch.no_grad()
def chunked_prefill(model, prompt, chunk_size):
    """把长 prompt 分块喂进去，每块复用前面累积的 KV。"""
    past, logits = None, None
    for start in range(0, prompt.size(1), chunk_size):
        piece = prompt[:, start:start + chunk_size]
        logits, past = model(piece, past_kvs=past, pos_offset=start)
    return logits, past


# 先验证分块和整段跑结果一致
prompt = torch.randint(0, model.cfg.vocab_size, (1, 1024), device=DEVICE)
whole_logits, _ = model(prompt)
chunk_logits, _ = chunked_prefill(model, prompt, chunk_size=256)

print(f"整段与分块的最后一位 logits 最大差异: "
      f"{(whole_logits[:, -1] - chunk_logits[:, -1]).abs().max().item():.2e}")
print(f"argmax 一致: {torch.equal(whole_logits[:, -1].argmax(-1), chunk_logits[:, -1].argmax(-1))}")

## 三、真实调度对比

跑一个真实的调度过程：8 条请求正在稳定解码，中途一条 2048 token 的长 prompt 插入。

In [ ]:
@torch.no_grad()
def episode(model, chunked, n_decode=8, dec_ctx=128, steps=24, long_len=2048, chunk=256):
    """返回 (每轮迭代耗时列表, 长请求 TTFT, 总耗时)。"""
    dec_ids = torch.randint(0, model.cfg.vocab_size, (n_decode, dec_ctx), device=DEVICE)
    logits, dec_past = model(dec_ids)
    dec_pos = dec_ctx
    dec_next = logits[:, -1].argmax(-1, keepdim=True)

    long_prompt = torch.randint(0, model.cfg.vocab_size, (1, long_len), device=DEVICE)
    n_chunks = math.ceil(long_len / chunk)
    chunk_idx = 0
    long_past = None
    long_ttft = None

    t_start = time.perf_counter()
    t_prev = t_start
    iters = []

    for step in range(steps):
        if chunked:
            # 每轮：一个 prefill 小块 + 一次 decode，混在一起跑
            if chunk_idx < n_chunks:
                start = chunk_idx * chunk
                piece = long_prompt[:, start:start + chunk]
                _, long_past = model(piece, past_kvs=long_past, pos_offset=start)
                chunk_idx += 1
                if chunk_idx == n_chunks:
                    long_ttft = time.perf_counter() - t_start
            logits, dec_past = model(dec_next, past_kvs=dec_past, pos_offset=dec_pos)
            dec_pos += 1
            dec_next = logits[:, -1].argmax(-1, keepdim=True)
        else:
            # 第一轮就把整段长 prefill 跑完，其他请求全部排队等它
            if step == 0:
                _, long_past = model(long_prompt)
                long_ttft = time.perf_counter() - t_start
            logits, dec_past = model(dec_next, past_kvs=dec_past, pos_offset=dec_pos)
            dec_pos += 1
            dec_next = logits[:, -1].argmax(-1, keepdim=True)

        sync()
        now = time.perf_counter()
        iters.append((now - t_prev) * 1000)
        t_prev = now

    return iters, long_ttft, time.perf_counter() - t_start


def pct(xs, q):
    s = sorted(xs)
    return s[min(len(s) - 1, int(q * len(s)))]


base_iters, base_ttft, base_total = episode(model, chunked=False)
chunk_iters, chunk_ttft, chunk_total = episode(model, chunked=True)

print("8 条稳定解码请求 + 1 条 2048 token 长 prompt 插入，共 24 轮迭代\n")
print(f"{'指标':<26}{'整段 prefill':>16}{'chunked prefill':>18}")
print("-" * 60)
print(f"{'解码请求 TBT p50 (ms)':<26}{pct(base_iters, 0.50):>16.1f}{pct(chunk_iters, 0.50):>18.1f}")
print(f"{'解码请求 TBT p95 (ms)':<26}{pct(base_iters, 0.95):>16.1f}{pct(chunk_iters, 0.95):>18.1f}")
print(f"{'解码请求 TBT 最大 (ms)':<26}{max(base_iters):>16.1f}{max(chunk_iters):>18.1f}")
print(f"{'长请求 TTFT (ms)':<26}{base_ttft * 1000:>16.1f}{chunk_ttft * 1000:>18.1f}")
print(f"{'24 轮总耗时 (ms)':<26}{base_total * 1000:>16.1f}{chunk_total * 1000:>18.1f}")

### 结果解读

大概率会看到这样一组数字：

| 现象 | 说明 |
|---|---|
| **TBT 最大值大幅下降** | 整段 prefill 时有一轮迭代耗时是其他轮的好几倍；分块后每轮都均匀 |
| **长请求 TTFT 变长** | 它要等好几轮才轮到结束，这是明确的代价 |
| **总耗时基本不变** | **总计算量没变**，只是把阻塞摊平了 |

这就是 chunked prefill 的本质：**它不是优化，是重新分配**。用长请求自己的 TTFT，换所有其他请求的尾延迟。

值不值得看业务：长请求占比低（比如 5%）而 decode 请求海量时，这笔交易非常划算——5% 的用户多等一点，95% 的用户不再卡顿。

## 四、什么时候不该用它

面试官喜欢追问边界条件，这三个都是真实的：

1. **chunk 切得太小**：每个 chunk 的矩阵乘太小，GPU 算力利用率骤降，吞吐反而变差。粒度要在"阻塞时间"和"算力效率"之间取平衡。
2. **显存压力大时**：分块 prefill 让更多请求同时处于"进行中"，KV cache 峰值占用上升，可能触发抢占。用显存换延迟，账要算清楚。
3. **本来就延迟不敏感**：离线批量推理只关心吞吐，chunked prefill 只带来额外调度开销。

还有一个容易忽略的：**它和 prefix caching 有重叠**。如果 prompt 大部分能命中缓存，prefill 本来就短，chunk 的意义就不大。两个优化不要重复投入。

## 五、面试话术

**问：chunked prefill 为什么能改善尾延迟？代价是什么？**

- **问题**：prefill 是不可中断的一次大计算，长 prompt 独占 GPU，让同批正在解码的请求排队，表现为 TBT 尖刺。
- **做法**：把 prefill 按 block 切块，每轮迭代算一块，与该轮 decode 混批。
- **收益**：把一次长阻塞摊成多次短阻塞，TBT p99 显著下降，延迟分布变得可预测。
- **代价**：长请求自身 TTFT 变长；chunk 过小降低算力效率；同时进行中的请求变多，KV 显存峰值上升。
- **本质**：总计算量不变，是延迟在请求之间的**重新分配**——用少数长请求的 TTFT 换整体的尾延迟。

最后那句"不是优化而是重新分配"是这句话的分水岭。它说明你知道自己在做什么交易，而不是在执行一个听说过名字的技术。

**作业**

1. 把 `chunk` 从 256 改成 64 和 1024，重跑 episode，观察 TBT 最大值 / TTFT / 总耗时三个指标怎么变，找出你机器上的平衡点。
2. 把 `n_decode` 从 8 改成 32，长 prefill 的阻塞效应是变强还是变弱？为什么？
3. 思考题：如果长请求的用户体验很重要（比如付费用户），设计什么机制既保住他的 TTFT 又保住其他人的尾延迟？（提示：优先级 + 抢占）

**下一章**：换个方向提速——用一个小模型给大模型"打草稿"，也就是投机解码。